<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 3 · Referencias que se copian bien
**Excel Corporativo · 12 Claves** — Relativas, absolutas y mixtas · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/03_referencias.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-03)

**Al terminar podrás:**
1. Ver qué calcula cada celda cuando una fórmula se copia con y sin $.
2. Reconocer en un informe el efecto de un total que no estaba fijo.
3. Entender por qué un valor de entrada va en una sola celda.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · Cuatro formas de la misma celda

| Escrita como | Al copiar hacia abajo | Al copiar a la derecha | Se usa para |
|---|---|---|---|
| `B5` | cambia la fila | cambia la columna | Lo normal: cada fila usa su propio dato |
| `$B$5` | no cambia | no cambia | Una tasa, un total, una fecha de corte |
| `$B5` | cambia la fila | **no** cambia | Una columna de datos que se lee desde varias columnas |
| `B$5` | **no** cambia | cambia la columna | Una fila de encabezados (escenarios, meses) |

El ejercicio de la hoja **Referencias**: proyectar la venta 2025 de cada sede con cuatro escenarios de crecimiento, escribiendo **una sola fórmula** en C5 y copiándola a toda la tabla.

In [ ]:
# @title Preparar la venta base y los escenarios
base = validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum().reindex(sedes.ciudad) / 1e6
esc = [-0.05, 0.0, 0.05, 0.10]
display(base.to_frame("Venta 2025 (millones)").style.format(lambda v: f"{v:,.0f}".replace(",", ".")))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué calcula cada celda si la fórmula se copia con estas referencias? |
> | **Qué se grafica** | Mapa de calor: sedes × escenarios · Celdas grises: la fórmula apunta a una celda equivocada |
> | **Por qué esta forma** | La tabla de doble entrada se ve como una cuadrícula; el gris marca exactamente dónde falla la copia. |
> | **Qué se descartó** | Solo el número de errores: no se vería el patrón (fila o columna equivocada). |
> | **Cómo se lee** | Con $B5 y C$4 toda la cuadrícula tiene color. Prueba las otras opciones y mira qué fila o columna se apaga. |
> | **En Excel 365** | Escribe la fórmula en C5 de la hoja Referencias y cópiala hasta F10. |

In [ ]:
# @title Gráfico 1 · Copiar la fórmula a toda la tabla
ref_base = "$B5"  # @param ["$B5", "B5", "$B$5", "B$5"]
ref_escenario = "C$4"  # @param ["C$4", "C4", "$C$4", "$C4"]
def apunta(ref, df, dc):
    col_fija, fila_fija = ref.startswith("$"), "$" in ref[1:]
    letra = ref.replace("$", "")[0]; fila = int(ref.replace("$", "")[1:])
    return (ord(letra) - 64 + (0 if col_fija else dc), fila + (0 if fila_fija else df))
z, txt, ok = [], [], 0
for i, s in enumerate(base.index):
    fz, ft = [], []
    for j, e in enumerate(esc):
        b, c = apunta(ref_base, i, j), apunta(ref_escenario, i, j)
        bien = b == (2, 5 + i) and c == (3 + j, 4)
        ok += bien
        fz.append(base[s] * (1 + e) if bien else np.nan)
        ft.append(f"{base[s] * (1 + e):,.0f}".replace(",", ".") if bien else f"✗ {chr(64 + b[0])}{b[1]}")
    z.append(fz); txt.append(ft)
fig = go.Figure(go.Heatmap(z=z, x=[f"{e:+.0%}" for e in esc], y=list(base.index), text=txt, texttemplate="%{text}",
                           colorscale=[[0, "#EEF2FF"], [1, AZUL]], showscale=False, hoverongaps=False))
fig.update_layout(title=f"={ref_base}*(1+{ref_escenario}) copiada: {ok} de 24 celdas correctas",
                  yaxis=dict(autorange="reversed"), height=360, plot_bgcolor="#E4E8F2")
fig.show()
display(Markdown("Las celdas grises muestran a qué celda apunta la venta base cuando la copia sale mal." if ok < 24 else "Toda la tabla quedó bien: la venta base fija en la columna B y el escenario fijo en la fila 4."))

## 2 · El total que no estaba fijo

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué ve el lector del informe cuando la participación se copió sin $? |
> | **Qué se grafica** | Dos paneles: participación por sede con =B5/B$11 y con =B5/B11 copiada |
> | **Por qué esta forma** | Lado a lado, la diferencia salta a la vista: el informe correcto y el que llega con errores. |
> | **Qué se descartó** | Explicar el error con texto: no se ve la consecuencia. |
> | **Cómo se lee** | A la derecha solo sobrevive la primera sede; las demás son #DIV/0! porque dividen por celdas vacías. |
> | **En Excel 365** | H5: =B5/B$11 y copiar hasta H10. |

In [ ]:
# @title Gráfico 2 · Participación con y sin $
from plotly.subplots import make_subplots
part = base / base.sum()
mal = pd.Series([part.iloc[0]] + [np.nan] * (len(part) - 1), index=part.index)
fig = make_subplots(rows=1, cols=2, subplot_titles=["=B5/B$11 (total fijo)", "=B5/B11 copiada (total que se mueve)"], shared_yaxes=True)
fig.add_bar(y=part.index, x=part.values, orientation="h", marker_color=AZUL, text=[porc(v) for v in part.values], textposition="outside", row=1, col=1, showlegend=False)
fig.add_bar(y=mal.index, x=mal.fillna(0).values, orientation="h", marker_color=MAGENTA, text=[porc(v) if v == v else "#DIV/0!" for v in mal.values], textposition="outside", row=1, col=2, showlegend=False)
fig.update_xaxes(tickformat=".0%", range=[0, part.max() * 1.6])
fig.update_layout(title="Con el total fijo, las seis sedes suman 100 %; sin él, el informe sale con cinco errores", yaxis=dict(autorange="reversed"), height=360)
fig.show()

## 3 · Un dato de entrada, una sola celda

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto cambia la venta con IVA si cambia la tasa? |
> | **Qué se grafica** | Barras: venta con IVA por sede, calculada con la tasa que elijas |
> | **Por qué esta forma** | Comparación de una medida entre sedes; al cambiar la tasa, todas las barras cambian a la vez, igual que en Excel con $K$1. |
> | **Qué se descartó** | Tasa escrita dentro de cada fórmula (*1,19): habría que corregir seis fórmulas. |
> | **Cómo se lee** | Cambia la tasa en la lista y mira que todo se recalcula desde un solo lugar. |
> | **En Excel 365** | J5: =B5*(1+$K$1) o =B5*(1+IVA) con un nombre definido. |

In [ ]:
# @title Gráfico 3 · Venta con IVA según la tasa
tasa = "19 %"  # @param ["19 %", "5 %", "0 %"]
t = int(tasa.split()[0]) / 100
con_iva = (base * (1 + t)).sort_values()
fig = go.Figure(go.Bar(x=con_iva.values, y=con_iva.index, orientation="h", marker_color=AZUL,
                       text=[f"$ {v:,.0f} M".replace(",", ".") for v in con_iva.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"Con IVA del {tasa}, la venta total sería {millones(con_iva.sum() * 1e6)}", xaxis_title=MM, height=340)
dar_aire(fig, con_iva.max())
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · En C5 está =B5/B11. ¿Qué fórmula queda en C6 al copiarla hacia abajo?
respuesta = "Elige una opción"  # @param ["Elige una opción", "=B5/B11", "=B6/B11", "=B6/B12", "=C6/C12"]
revisar(respuesta, "=B6/B12", """Las dos referencias son relativas y bajan una fila. B12 está vacía: #DIV/0!.""")

In [ ]:
# @title Reto 2 · ¿Qué fórmula en C5 llena bien toda la tabla de escenarios?
respuesta = "Elige una opción"  # @param ["Elige una opción", "=B5*(1+C4)", "=$B$5*(1+$C$4)", "=$B5*(1+C$4)", "=B$5*(1+$C4)"]
revisar(respuesta, "=$B5*(1+C$4)", """Columna de la venta base fija ($B) y fila de escenarios fija (C$4): 24 de 24 en el gráfico 1.""")

In [ ]:
# @title Reto 3 · La tasa de IVA está en $K$1. Si cambia, ¿cuántas celdas hay que modificar?
respuesta = "Elige una opción"  # @param ["Elige una opción", "1", "6", "24", "Todas las fórmulas"]
revisar(respuesta, "1", """Esa es la razón de poner los datos de entrada en una celda propia.""")

## Apóyate en IA
Si una fórmula copiada da resultados raros, pide que te expliquen la copia celda por celda:

```text
En Excel escribí =B5/B11 en C5 y la copié hasta C10. Dime qué fórmula queda en cada celda de C6 a C10,
qué resultado da cada una y cómo corregirla con $ o con un nombre definido.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Al copiar, las referencias se mueven: el $ fija lo que no debe moverse.
* Pregunta: ¿debe cambiar al copiar hacia abajo? ¿y a la derecha?
* Los datos de entrada (tasas, totales, fechas de corte) van en una sola celda.

**Siguiente:** [Clave 4 · Lógica condicional legible](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-04) · [Cuaderno 04](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/04_logica_condicional.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>